<a href="https://colab.research.google.com/github/weagan/LAYA-JEV/blob/main/laya_typed_decisions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laya Typed Decisions — Google Colab

This version follows the current Hugging Face quickstart for `convaiinnovations/laya-typed-decisions`.
The model is loaded through the `laya` SDK rather than `AutoModelForCausalLM`/Llama tokenization.


In [1]:
# Install the Laya SDK
%pip install -q -U laya


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.9/128.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.0/360.0 kB 12.0 MB/s eta 0:00:00


In [2]:
# If model loading hangs in Colab, prevent Transformers from probing TensorFlow.
import os
os.environ["USE_TF"] = "0"

import laya

model_id = "convaiinnovations/laya-typed-decisions"
agent = laya.load(model_id)
print("Loaded:", model_id)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loaded: convaiinnovations/laya-typed-decisions


/usr/local/lib/python3.13/dist-packages/laya/agent.py:2125: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(model_id_or_path, device=device, token=token, subfolder=subfolder, fast=fast,


In [3]:
questions = {
    "department": {
        "type": "choice",
        "instructions": "Which department should handle this ticket?",
        "criteria": {
            "billing": "Invoices, subscription issues, duplicate payments, and refund requests.",
            "technical": "Bugs, error codes, service outages, API integration, and system issues.",
            "other": "General questions, feature suggestions, or feedback."
        }
    },
    "urgency": {
        "type": "score",
        "instructions": "Assess the urgency of this customer inquiry.",
        "criteria": ["low", "medium", "high"]
    },
    "churn_risk": {
        "type": "noul",
        "instructions": "Is there an indication or a direct threat that the customer is planning to leave or cancel their service?"
    }
}

tests = [
    ("Billing & Churn Risk", "Hi, we were billed twice for March. Please refund the duplicate today or we will cancel our plan."),
    ("Technical Bug", "The dashboard page is showing a 500 internal server error when I try to export my weekly analytics report. Please help!"),
    ("General Query", "I was wondering if there are any plans to support dark mode on the mobile application in the future?"),
    ("Critical Outage & Churn Risk", "Our entire production site is down right now! If this is not resolved within the hour, we will be moving to a competitor.")
]


In [4]:
for title, state in tests:
    print("=" * 70)
    print(title)
    print("Input:", state)
    print("=" * 70)

    result = agent.predict(state, questions)
    print(result)
    print()


Billing & Churn Risk
Input: Hi, we were billed twice for March. Please refund the duplicate today or we will cancel our plan.
{'model': 'laya-rl-agent', 'answers': {'department': {'type': 'choice', 'choice': 'billing', 'probabilities': {'billing': 0.8918, 'technical': 0.0529, 'other': 0.0554}, 'confidence': 0.6197, 'answer_confidence': 0.8918, 'action': {'act_probability': 1.0}}, 'urgency': {'type': 'score', 'score': 1.7896, 'legend': {'0': 'low', '1': 'medium', '2': 'high'}, 'probabilities': {'0': 0.0361, '1': 0.1382, '2': 0.8257}, 'confidence': 0.498, 'answer_confidence': 0.8257, 'action': {'act_probability': 1.0}}, 'churn_risk': {'type': 'noul', 'noul': 0.7373, 'confidence': 0.7373, 'answer_confidence': 0.7373, 'action': {'act_probability': 1.0}}}, 'usage': {'input_tokens': 203, 'output_tokens': 0, 'state_tokens': 21, 'state_tokens_dropped': 0, 'truncated': False, 'truncated_questions': []}}

Technical Bug
Input: The dashboard page is showing a 500 internal server error when I try t

## Notes

- No Hugging Face token is required for this public model unless your environment otherwise requires authentication.
- This checkpoint is English-only and specialized for four typed-decision workflows.
- The model card warns that probabilities are not currently well calibrated; treat confidence values accordingly.
